## 自定义模型有效性
通过相同的数据集进行forward推理，如果lerobot_policy_pi05 与lerobot.policies.pi05 推理结果相同，证明模型架构有效

6.8日版本，后续会修改lerobot_policy_pi05，此文档仅记录，可通过git回溯

In [1]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"  # 禁止联网，只用本地 cache

from lerobot.datasets.lerobot_dataset import LeRobotDataset

dataset = LeRobotDataset(
    repo_id='/vla/.data/test',
    video_backend='torchcodec',
)
dataset

/mnt/workspace/luyi/.cache/miniconda3/envs/myvla/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LeRobotDataset({
    Repository ID: '/vla/.data/test',
    Number of selected episodes: '1',
    Number of selected samples: '436',
    Features: '['observation.state', 'action', 'observation.images.robot0_agentview_left_image', 'observation.images.robot0_agentview_right_image', 'observation.images.robot0_eye_in_hand_image', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']',
})',

### 模型加载

In [2]:
import torch
from lerobot.utils.import_utils import register_third_party_plugins
from lerobot.policies.factory import make_pre_post_processors

from lerobot_policy_pi05 import PI05Policy as My_PI05Policy
from lerobot.policies.pi05 import PI05Policy as Official_PI05Policy

model_id = "/vla/.models/lerobot-pi05_base"
register_third_party_plugins()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

my_policy = My_PI05Policy.from_pretrained(model_id).to(device).eval()
official_policy = Official_PI05Policy.from_pretrained(model_id).to(device).eval()

/mnt/workspace/luyi/.cache/miniconda3/envs/myvla/lib/python3.10/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


The PI05 model is a direct port of the OpenPI implementation. 
This implementation follows the original OpenPI structure for compatibility. 
Original implementation: https://github.com/Physical-Intelligence/openpi


Loading model from: /vla/.models/lerobot-pi05_base
✓ Loaded state dict from model.safetensors
Remapped: action_in_proj.bias -> model.action_in_proj.bias
Remapped: action_in_proj.weight -> model.action_in_proj.weight
Remapped: action_out_proj.bias -> model.action_out_proj.bias
Remapped: action_out_proj.weight -> model.action_out_proj.weight
Remapped: paligemma_with_expert.gemma_expert.lm_head.weight -> model.paligemma_with_expert.gemma_expert.lm_head.weight
Remapped: paligemma_with_expert.gemma_expert.model.layers.0.input_layernorm.dense.bias -> model.paligemma_with_expert.gemma_expert.model.layers.0.input_layernorm.dense.bias
Remapped: paligemma_with_expert.gemma_expert.model.layers.0.input_layernorm.dense.weight -> model.paligemma_with_expert.gemma_expert.model.layers.0.input_layernorm.dense.weight
Remapped: paligemma_with_expert.gemma_expert.model.layers.0.mlp.down_proj.weight -> model.paligemma_with_expert.gemma_expert.model.layers.0.mlp.down_proj.weight
Remapped: paligemma_with_exp

The PI05 model is a direct port of the OpenPI implementation. 
This implementation follows the original OpenPI structure for compatibility. 
Original implementation: https://github.com/Physical-Intelligence/openpi


Loading model from: /vla/.models/lerobot-pi05_base
✓ Loaded state dict from model.safetensors
Remapped: action_in_proj.bias -> model.action_in_proj.bias
Remapped: action_in_proj.weight -> model.action_in_proj.weight
Remapped: action_out_proj.bias -> model.action_out_proj.bias
Remapped: action_out_proj.weight -> model.action_out_proj.weight
Remapped: paligemma_with_expert.gemma_expert.lm_head.weight -> model.paligemma_with_expert.gemma_expert.lm_head.weight
Remapped: paligemma_with_expert.gemma_expert.model.layers.0.input_layernorm.dense.bias -> model.paligemma_with_expert.gemma_expert.model.layers.0.input_layernorm.dense.bias
Remapped: paligemma_with_expert.gemma_expert.model.layers.0.input_layernorm.dense.weight -> model.paligemma_with_expert.gemma_expert.model.layers.0.input_layernorm.dense.weight
Remapped: paligemma_with_expert.gemma_expert.model.layers.0.mlp.down_proj.weight -> model.paligemma_with_expert.gemma_expert.model.layers.0.mlp.down_proj.weight
Remapped: paligemma_with_exp

In [3]:
my_preprocess, my_postprocess = make_pre_post_processors(
    my_policy.config,
    dataset_stats=dataset.meta.stats,
    preprocessor_overrides={"device_processor": {"device": str(device)}},
)

official_preprocess, official_postprocess = make_pre_post_processors(
    official_policy.config,
    dataset_stats=dataset.meta.stats,
    preprocessor_overrides={"device_processor": {"device": str(device)}},
)

### 推理验证

> `select_action` 每次会随机采样 noise，两次结果不同是正常现象，不能直接用来比等价性。

In [4]:
def to_device(batch):
    return {
        k: v.to(device, non_blocking=True) if isinstance(v, torch.Tensor) else v
        for k, v in batch.items()
    }

batch = to_device(my_preprocess(dataset[0]))

In [5]:
with torch.inference_mode():
    pred_action_raw = my_policy.select_action(batch)
pred_action_raw

tensor([[-0.5169,  0.9697,  0.5261,  0.8694,  0.5457, -0.5005, -0.9667,  0.2266,
         -0.0290,  0.0255,  0.0268, -0.0176,  0.0723,  0.0646,  0.4629,  0.2541,
          0.0300,  0.0577, -0.0593, -0.0428,  0.0422,  0.1107, -0.0146,  0.0768,
          0.0328, -0.0286,  0.0449, -0.0040, -0.0048,  0.0108, -0.0637,  0.0477]],
       device='cuda:0')

In [6]:
batch = to_device(official_preprocess(dataset[0]))
with torch.inference_mode():
    pred_action_raw = official_policy.select_action(batch)
pred_action_raw

tensor([[-0.5563,  1.0626,  0.5692,  0.8870,  0.5155, -0.5587, -0.8825,  0.2541,
         -0.0049,  0.0379,  0.0436, -0.0380,  0.0352,  0.0705,  0.5096,  0.5220,
          0.0610,  0.2221,  0.0102, -0.0919,  0.0419,  0.0730,  0.0049,  0.0488,
          0.0136, -0.0307,  0.0269, -0.0050, -0.0030,  0.0138, -0.0431,  0.0447]],
       device='cuda:0')

### 等价验证

正确对比方式（每步输出 PASS / FAIL）：
1. 比权重 state_dict
2. 比 forward loss（固定 noise + time）
3. 比 predict_action_chunk（固定 noise）

In [7]:
from lerobot.datasets.factory import resolve_delta_timestamps

print("1/3 resolve_delta_timestamps ...")
delta_ts = resolve_delta_timestamps(my_policy.config, dataset.meta)
print("   delta_ts keys:", list(delta_ts.keys()) if delta_ts else None)

print("2/3 构建 dataset_train（含 chunk action）...")
dataset_train = LeRobotDataset(
    repo_id="/vla/.data/test",
    delta_timestamps=delta_ts,
    video_backend="torchcodec",
)

print("3/3 读取 dataset_train[0] + preprocess（含视频解码，可能需 10~30s）...")
sample = dataset_train[0]
print("   raw action shape:", sample["action"].shape)  # 期望 [50, 12]

batch_eq = to_device(my_preprocess(sample))

# preprocess 有时不会给 chunk action 加 batch 维，需手动 [50,D] -> [1,50,D]
if batch_eq["action"].ndim == 2:
    batch_eq["action"] = batch_eq["action"].unsqueeze(0)

action_dim = my_policy.config.output_features["action"].shape[0]
print("batch_eq action shape:", batch_eq["action"].shape)  # 期望 [1, 50, 12]

1/3 resolve_delta_timestamps ...
   delta_ts keys: ['action']
2/3 构建 dataset_train（含 chunk action）...
3/3 读取 dataset_train[0] + preprocess（含视频解码，可能需 10~30s）...
   raw action shape: torch.Size([50, 12])
batch_eq action shape: torch.Size([1, 50, 12])


#### Step 1 — 比权重

In [8]:
def compare_weights(m1, m2, atol=0.0):
    s1, s2 = m1.state_dict(), m2.state_dict()
    only_my = sorted(set(s1) - set(s2))
    only_off = sorted(set(s2) - set(s1))
    diffs = [(k, (s1[k].float() - s2[k].float()).abs().max().item())
             for k in sorted(set(s1) & set(s2))
             if (s1[k].float() - s2[k].float()).abs().max().item() > atol]
    ok = not only_my and not only_off and not diffs
    print("=" * 50)
    print("Step 1: 权重对比")
    print(f"  my={len(s1)} keys, official={len(s2)} keys")
    print(f"  仅 my 有 ({len(only_my)}):", only_my[:3], "..." if len(only_my) > 3 else "")
    print(f"  仅 official 有 ({len(only_off)}):", only_off[:3], "..." if len(only_off) > 3 else "")
    print(f"  共有但数值不同 ({len(diffs)}):", [k for k, _ in diffs[:3]], "..." if len(diffs) > 3 else "")
    if diffs:
        print(f"  最大 diff: {diffs[0][0]} = {diffs[0][1]:.6e}")
    print("  >>>", "PASS ✓" if ok else "FAIL ✗")
    return ok

step1_ok = compare_weights(my_policy, official_policy)

Step 1: 权重对比
  my=813 keys, official=813 keys
  仅 my 有 (0): [] 
  仅 official 有 (0): [] 
  共有但数值不同 (0): [] 
  >>> PASS ✓


#### Step 2 — 比 forward loss（固定 noise + time）

In [9]:
from lerobot.utils.constants import OBS_LANGUAGE_ATTENTION_MASK, OBS_LANGUAGE_TOKENS

def forward_loss(policy, batch, noise, time):
    images, img_masks = policy._preprocess_images(batch)
    tokens = batch[OBS_LANGUAGE_TOKENS]
    masks = batch[OBS_LANGUAGE_ATTENTION_MASK]
    actions = policy.prepare_action(batch)
    losses = policy.model.forward(images, img_masks, tokens, masks, actions, noise=noise, time=time)
    return losses[:, :, :action_dim].mean()

actions = my_policy.prepare_action(batch_eq)
print("prepare_action shape:", actions.shape)  # 期望 [1, 50, 32]

noise = my_policy.model.sample_noise(actions.shape, device)
time = my_policy.model.sample_time(actions.shape[0], device)

with torch.inference_mode():
    loss_my = forward_loss(my_policy, batch_eq, noise, time)
    loss_off = forward_loss(official_policy, batch_eq, noise, time)

loss_diff = (loss_my - loss_off).abs().item()
step2_ok = loss_diff < 1e-5

print("=" * 50)
print("Step 2: forward loss（同一 noise / time）")
print(f"  my loss      = {loss_my.item():.8f}")
print(f"  official loss= {loss_off.item():.8f}")
print(f"  |diff|       = {loss_diff:.2e}")
print("  >>>", "PASS ✓" if step2_ok else "FAIL ✗")

prepare_action shape: torch.Size([1, 50, 32])
Step 2: forward loss（同一 noise / time）
  my loss      = 0.56468296
  official loss= 0.56468296
  |diff|       = 0.00e+00
  >>> PASS ✓


#### Step 3 — 比 predict_action_chunk（固定 noise）

In [10]:
cfg = my_policy.config
fixed_noise = torch.randn(1, cfg.chunk_size, cfg.max_action_dim, device=device, dtype=torch.float32)

with torch.inference_mode():
    chunk_my = my_policy.predict_action_chunk(batch_eq, noise=fixed_noise.clone())
    chunk_off = official_policy.predict_action_chunk(batch_eq, noise=fixed_noise.clone())

action_my = chunk_my[0, 0, :action_dim]
action_off = chunk_off[0, 0, :action_dim]
action_diff = (action_my - action_off).abs().max().item()
step3_ok = action_diff < 1e-4

print("=" * 50)
print("Step 3: predict_action_chunk（固定 noise）")
print(f"  action[0,:3] my      = {action_my[:3].tolist()}")
print(f"  action[0,:3] official= {action_off[:3].tolist()}")
print(f"  max |diff|           = {action_diff:.2e}")
print("  >>>", "PASS ✓" if step3_ok else "FAIL ✗")

Step 3: predict_action_chunk（固定 noise）
  action[0,:3] my      = [-0.5281607508659363, 1.0795261859893799, 0.5218471884727478]
  action[0,:3] official= [-0.5281607508659363, 1.0795261859893799, 0.5218471884727478]
  max |diff|           = 0.00e+00
  >>> PASS ✓


#### 汇总

In [11]:
results = {"Step1 权重": step1_ok, "Step2 forward loss": step2_ok, "Step3 推理(chunk)": step3_ok}
print("=" * 50)
print("汇总")
for name, ok in results.items():
    print(f"  {name}: {'PASS ✓' if ok else 'FAIL ✗'}")
print("=" * 50)
print("整体:", "全部 PASS ✓ — 架构等价" if all(results.values()) else "存在 FAIL ✗ — 见上方详情")

if not step1_ok:
    print("\n提示: Step1 失败常见于 checkpoint 缺少 embed_tokens，两个实例随机初始化不同。")

汇总
  Step1 权重: PASS ✓
  Step2 forward loss: PASS ✓
  Step3 推理(chunk): PASS ✓
整体: 全部 PASS ✓ — 架构等价
